# TAP VOParquet Query

## Imports

In [ ]:
import io
import os

import requests
from astropy.table import Table
from lsst.rsp import get_tap_service

## Service Instantiation

In [ ]:
service = get_tap_service("tap")
assert service is not None
print(f"TAP service URL: {service.baseurl}")

In [ ]:
token = os.environ["ACCESS_TOKEN"]
session = requests.Session()
session.headers["Authorization"] = f"Bearer {token}"

## Async Job with VOParquet Response

In [ ]:
query = """SELECT TOP 50000 * FROM dp1.Object"""
job = service.submit_job(query, responseformat="application/vnd.apache.parquet")
print(f"Job submitted with ID: {job.job_id}")
job.run()
print(f"Job started, phase: {job.phase}")

## Wait for Completion

In [ ]:
job.wait(phases=["COMPLETED", "ERROR"], timeout=300)
print(f"Job phase: {job.phase}")
if job.phase == "ERROR":
    job.raise_if_error()
assert job.phase == "COMPLETED", f"Job failed with phase: {job.phase}"

## Fetch Results

In [ ]:
result_url = f"{job.url}/results/result"
print(f"Fetching result from: {result_url}")
response = session.get(result_url)
response.raise_for_status()
print(f"Content length: {len(response.content):,} bytes")

## Parse with Astropy

Load the VOParquet file into an Astropy Table and verify the row count.

In [ ]:
buf = io.BytesIO(response.content)
table = Table.read(buf, format="parquet.votable")
print(f"Retrieved {len(table):,} objects via VOParquet")
assert len(table) >= 50000, f"Expected >= 50 000 rows, got {len(table)}"
table[:5]

## Cleanup

In [ ]:
job.delete()
print("Job deleted")